# Piyu AI Fashion Design Generator — Google Colab

**Pipeline:** RealVisXL → SAM → IDM-VTON

This notebook is designed for a Colab NVIDIA T4/L4 GPU. It uses your updated repository and avoids re-downloading model files when they already exist in the runtime.

> **Important:** Colab runtimes are temporary. Large model files can disappear when the runtime resets.


In [ ]:
# 1. GPU / environment check
import torch, sys, os
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2), "GB")
else:
    raise RuntimeError("Enable Runtime → Change runtime type → GPU.")


In [ ]:
# 2. Install dependencies
!pip install -q "numpy==1.26.4" "diffusers>=0.29.0" "transformers>=4.41.0" "accelerate>=0.30.0" "safetensors>=0.4.3" "tokenizers>=0.19.0,<0.20" "gradio>=4.0.0" "onnxruntime==1.20.1" "auto1111sdk"
!pip install -q git+https://github.com/facebookresearch/segment-anything.git


## 3. Clone your updated project

The notebook uses your updated repository:

`Piyu242005/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026`

If the repository already exists, it is reused.


In [ ]:
# 3. Clone / locate your updated repository
from pathlib import Path
import subprocess, os

REPO_URL = "https://github.com/Piyu242005/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026.git"
REPO_ROOT = Path("/content/AI-Fashion-Design-Generator-IBM-INTERSHIP-2026")
PROJECT = REPO_ROOT / "Piyu-AI-Clothing-Fashion-Design-Generator"

if not PROJECT.exists():
    if not REPO_ROOT.exists():
        !git clone {REPO_URL} {REPO_ROOT}
    else:
        print("Repository root already exists.")
else:
    print("Updated project already exists.")

%cd {PROJECT}
print("Project:", Path.cwd())
print("Files:", [p.name for p in Path.cwd().iterdir()])


In [ ]:
# 4. Clone IDM-VTON inside the project if needed
from pathlib import Path
idm = Path("idm_vton")
if not idm.exists():
    !git clone https://github.com/yisol/IDM-VTON.git idm_vton
else:
    print("idm_vton already exists.")

# Required directories
for d in [
    Path("weights"),
    Path("idm_vton/ckpt/densepose"),
    Path("idm_vton/ckpt/humanparsing"),
    Path("idm_vton/ckpt/openpose/ckpts"),
    Path("reference_images"),
    Path("results"),
    Path("samples"),
]:
    d.mkdir(parents=True, exist_ok=True)

print("IDM-VTON ready.")


## 5. Download model checkpoints

The notebook first checks whether the files already exist anywhere under `/content`. If found, it links them into the current project.

If they are not found, the cells download them.


In [ ]:
# 5A. Find or download RealVisXL + SAM
from pathlib import Path
import os, glob

def first_match(name):
    hits = list(Path("/content").rglob(name))
    return hits[0] if hits else None

# RealVisXL
realvis = first_match("realvisxl.safetensors")
if realvis is None:
    print("Downloading RealVisXL (~6.5 GB)...")
    !wget -O weights/realvisxl.safetensors "https://civitai.com/api/download/models/361593?type=Model&format=SafeTensor&size=pruned&fp=fp16"
else:
    print("Found existing RealVisXL:", realvis)
    dst = Path("weights/realvisxl.safetensors")
    if not dst.exists():
        os.symlink(realvis, dst)

# SAM
sam = first_match("sam_vit_h_4b8939.pth")
if sam is None:
    print("Downloading SAM ViT-H (~2.4 GB)...")
    !wget -O weights/sam_vit_h_4b8939.pth "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth"
else:
    print("Found existing SAM:", sam)
    dst = Path("weights/sam_vit_h_4b8939.pth")
    if not dst.exists():
        os.symlink(sam, dst)

print("Model files ready.")


In [ ]:
# 5B. Download IDM-VTON supporting checkpoints
downloads = [
    ("idm_vton/ckpt/densepose/model_final_162be9.pkl",
     "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/densepose/model_final_162be9.pkl?download=true"),
    ("idm_vton/ckpt/humanparsing/parsing_atr.onnx",
     "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/humanparsing/parsing_atr.onnx?download=true"),
    ("idm_vton/ckpt/humanparsing/parsing_lip.onnx",
     "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/humanparsing/parsing_lip.onnx?download=true"),
    ("idm_vton/ckpt/openpose/ckpts/body_pose_model.pth",
     "https://huggingface.co/spaces/yisol/IDM-VTON/resolve/main/ckpt/openpose/ckpts/body_pose_model.pth?download=true"),
]
for dst, url in downloads:
    if not Path(dst).exists():
        print("Downloading:", dst)
        !wget -O "$dst" "$url"
    else:
        print("Already exists:", dst)


In [ ]:
# 5C. Ensure IDM-VTON image encoder / IP-Adapter files exist
# These are included in the IDM-VTON repository in the current checkout.
!find idm_vton/ckpt -maxdepth 3 -type f | sort


## 6. Fix the RealVisXL filename expected by your script

Your `generate_model.py` expects:

`weights/realvisxlV40_v40LightningBakedvae.safetensors`

while the downloaded file is named `realvisxl.safetensors`.


In [ ]:
# 6. Create a compatible filename link
from pathlib import Path
import os
src = Path("weights/realvisxl.safetensors")
dst = Path("weights/realvisxlV40_v40LightningBakedvae.safetensors")

if src.exists() and not dst.exists():
    os.symlink(src.resolve(), dst)
    print("Created:", dst)
elif dst.exists():
    print("Expected filename already exists.")
else:
    raise FileNotFoundError(src)

!ls -lh weights


## 7. Generate the AI fashion model

Default prompt can be changed below.


In [ ]:
# 7. Generate a realistic model wearing the requested clothing
PROMPT = "a modern red oversized fashion jacket, professional fashion photography"
MODEL_IMAGE = "reference_images/fashion_model.png"

!python generate_model.py --prompt "$PROMPT" --output_path "$MODEL_IMAGE"


In [ ]:
# 7B. Display generated model
from IPython.display import display
from PIL import Image
display(Image.open(MODEL_IMAGE))


## 8. Create a clothing mask with SAM

The original `segment.py` opens a desktop Matplotlib window and waits for three clicks. That is not reliable in Colab.

This cell provides a **Colab-safe coordinate-based SAM segmentation**. First display the image with axes, then enter three positive points on the garment.


In [ ]:
# 8A. Display image with coordinates
import matplotlib.pyplot as plt
from PIL import Image

img = Image.open(MODEL_IMAGE).convert("RGB")
print("Image size:", img.size, "(width, height)")

plt.figure(figsize=(9,12))
plt.imshow(img)
plt.xlim(0, img.width)
plt.ylim(img.height, 0)
plt.grid()
plt.title("Use this image to estimate 3 points INSIDE the clothing")
plt.show()


In [ ]:
# 8B. Enter three clothing points
# Example format: 380,420
points = []
for i in range(3):
    raw = input(f"Point {i+1} (x,y): ").strip()
    x, y = [int(v.strip()) for v in raw.split(",")]
    points.append([x, y])

print("Selected points:", points)


In [ ]:
# 8C. Run SAM using the three selected points
import numpy as np
import torch, cv2
from segment_anything import SamPredictor, sam_model_registry
from PIL import Image
import matplotlib.pyplot as plt

sam = sam_model_registry["vit_h"](checkpoint="weights/sam_vit_h_4b8939.pth")
sam.to(device="cuda")
predictor = SamPredictor(sam)

rgb = np.array(Image.open(MODEL_IMAGE).convert("RGB"))
predictor.set_image(rgb)

input_point = np.array(points, dtype=np.float32)
input_label = np.ones(len(points), dtype=np.int32)

masks, scores, logits = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True,
)

best = int(np.argmax(scores))
mask = masks[best]

MASK_PATH = "reference_images/fashion_model_mask.jpg"
mask_u8 = (mask.astype(np.uint8) * 255)
cv2.imwrite(MASK_PATH, mask_u8)

print("Mask saved:", MASK_PATH)
print("SAM score:", float(scores[best]))

plt.figure(figsize=(9,12))
plt.imshow(rgb)
plt.imshow(mask, alpha=0.45)
plt.scatter(input_point[:,0], input_point[:,1], c="red", s=80)
plt.axis("off")
plt.show()

del sam, predictor
torch.cuda.empty_cache()


## 9. Prepare a garment image

Upload a garment image such as a jacket, shirt, dress, or crop top.

Use a clean product-style image with the garment clearly visible.


In [ ]:
# 9. Upload garment image
from google.colab import files
uploaded = files.upload()

if not uploaded:
    raise RuntimeError("No garment image uploaded.")

uploaded_name = next(iter(uploaded))
GARMENT_PATH = "samples/garment.png"

from shutil import copyfile
copyfile(uploaded_name, GARMENT_PATH)

print("Garment saved as:", GARMENT_PATH)


In [ ]:
# 9B. Display garment
display(Image.open(GARMENT_PATH).convert("RGB"))


## 10. Prepare IDM-VTON inference

The original `try_on.py` expects the IDM-VTON source modules and downloads the main `yisol/idm_vton` model from Hugging Face.

We add the repository paths to Python so the imports resolve correctly in Colab.


In [ ]:
# 10. Configure Python paths for IDM-VTON
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
IDM = PROJECT / "idm_vton"

for p in [
    str(IDM),
    str(IDM / "gradio_demo"),
    str(IDM / "preprocess"),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("IDM-VTON paths configured.")


## 11. Run virtual try-on

This step may download the main `yisol/idm_vton` checkpoint from Hugging Face on its first run.

The T4 has 14.6 GB VRAM, so CUDA memory is limited. If you receive an out-of-memory error, reduce other GPU usage and restart the runtime before retrying.


In [ ]:
# 11. Run IDM-VTON
CLOTH_TYPE = "jacket"
FINAL_IMAGE = "results/final_tryon.png"

!python try_on.py     --reference_image "$MODEL_IMAGE"     --mask "$MASK_PATH"     --garment "$GARMENT_PATH"     --cloth_type "$CLOTH_TYPE"     --output_path "$FINAL_IMAGE"


In [ ]:
# 11B. Display final result
from IPython.display import display
from PIL import Image

if Path(FINAL_IMAGE).exists():
    display(Image.open(FINAL_IMAGE))
else:
    print("Final image was not created. Check the error from the previous cell.")


# ✅ Pipeline complete

Expected output:

**Text prompt → RealVisXL model → SAM garment mask → IDM-VTON virtual try-on**

Generated files:

- `reference_images/fashion_model.png`
- `reference_images/fashion_model_mask.jpg`
- `samples/garment.png`
- `results/final_tryon.png`
